In [7]:
import os
import cv2
from pathlib import Path
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import kagglehub

In [20]:
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, remove_small_holes

In [2]:
# Descarga del dataset 
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")
print("Path to dataset files:", path)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 663M/663M [02:42<00:00, 4.27MB/s]

Extracting files...


Path to dataset files: /home/xaraxx/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3


In [3]:
base_path = Path(path) / "TB_Chest_Radiography_Database"
normal_path = base_path / "Normal"
tb_path = base_path / "Tuberculosis"

normal_imgs = list(normal_path.glob("*.png"))
tb_imgs = list(tb_path.glob("*.png"))

print(f"Normal: {len(normal_imgs)}")
print(f"Tuberculosis: {len(tb_imgs)}")

Normal: 3500
Tuberculosis: 700


In [4]:
data = []

for img_path in normal_imgs:

    img = Image.open(img_path)

    data.append({
        "Nombre_Archivo": img_path.name,
        "path": str(img_path),
        "clase": "Normal",
        "Ancho": img.width,
        "Alto": img.height,
        "Modo": img.mode
    })

for img_path in tb_imgs:

    img = Image.open(img_path)

    data.append({
        "Nombre_Archivo": img_path.name,
        "path": str(img_path),
        "clase": "Tuberculosis",
        "Ancho": img.width,
        "Alto": img.height,
        "Modo": img.mode
    })

df = pd.DataFrame(data)

In [5]:
df


,Nombre_Archivo,path,clase,Ancho,Alto,Modo
0,Normal-36.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
1,Normal-332.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
2,Normal-840.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
3,Normal-1965.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
4,Normal-1304.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
...,...,...,...,...,...,...
4195,Tuberculosis-366.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB
4196,Tuberculosis-555.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB
4197,Tuberculosis-667.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,L
4198,Tuberculosis-217.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB


In [8]:
def calcular_glcm_dataset(df):
    """
    Calcula las características GLCM promedio para cada imagen 
    """
    glcm_features_list = []
    
    for idx, fila in df.iterrows():
        try:
            # 1. Leer la imagen directamente en escala de grises (maneja RGB y L)
            ruta_img = str(fila['path'])
            img = cv2.imread(ruta_img, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                print(f"Error: No se pudo cargar la imagen en la fila {idx}: {ruta_img}")
                # Insertar NaNs si la imagen está corrupta o no existe
                glcm_features_list.append({
                    "glcm_contrast": np.nan, "glcm_dissimilarity": np.nan,
                    "glcm_homogeneity": np.nan, "glcm_energy": np.nan, "glcm_correlation": np.nan
                })
                continue
            
            # Asegurar formato uint8 necesario para graycomatrix
            img_uint8 = img.astype(np.uint8)
            
            # 2. Calcular la matriz GLCM (distancia 1, 4 ángulos)
            glcm = graycomatrix(img_uint8, distances=[1],
                                angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                                levels=256, symmetric=True, normed=True)
            
            # 3. Extraer las propiedades promediando los 4 ángulos
            features = {
                "glcm_contrast": graycoprops(glcm, "contrast").mean(),
                "glcm_dissimilarity": graycoprops(glcm, "dissimilarity").mean(),
                "glcm_homogeneity": graycoprops(glcm, "homogeneity").mean(),
                "glcm_energy": graycoprops(glcm, "energy").mean(),
                "glcm_correlation": graycoprops(glcm, "correlation").mean()
            }
            
            glcm_features_list.append(features)
            
        except Exception as e:
            print(f"Error inesperado en la fila {idx}: {e}")
            glcm_features_list.append({
                "glcm_contrast": np.nan, "glcm_dissimilarity": np.nan,
                "glcm_homogeneity": np.nan, "glcm_energy": np.nan, "glcm_correlation": np.nan
            })
            
    # 4. Convertir la lista de características en un DataFrame
    df_features = pd.DataFrame(glcm_features_list)
    
    # 5. Concatenar horizontalmente con tu DataFrame original
    df_resultado = pd.concat([df.reset_index(drop=True), df_features], axis=1)
    
    return df_resultado

In [21]:
def calcular_caracteristicas_forma_dataset(df):
    """
    Segmenta la región más densa de cada imagen en la columna 'path'
    y extrae sus características morfológicas de forma segura.
    """
    shape_features_list = []
    
    for idx, fila in df.iterrows():
        try:
            # 1. Leer imagen en escala de grises
            ruta_img = str(fila['path'])
            img = cv2.imread(ruta_img, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                print(f"[-] Fila {idx}: No se pudo cargar la imagen en la ruta: {ruta_img}")
                raise ValueError("Ruta inválida o archivo corrupto")
                
            img_uint8 = img.astype(np.uint8)
            
            # 2. Segmentación con Otsu (con respaldo si se queda vacía)
            thresh = threshold_otsu(img_uint8)
            binary = img_uint8 > (thresh * 1.05)
            
            # Si la optimización del 5% borra toda la segmentación, usamos Otsu normal
            if not np.any(binary):
                binary = img_uint8 > thresh
                
            # Asegurar tipo bool para evitar conflictos en scikit-image
            binary = binary.astype(bool)
            binary = remove_small_objects(binary, min_size=80)
            binary = remove_small_holes(binary, area_threshold=200)
            
            labeled = label(binary)
            regions = regionprops(labeled, intensity_image=img_uint8)
            
            # Si aún así no hay regiones detectadas
            if not regions:
                shape_features_list.append({
                    "shape_area": np.nan, "shape_perimeter": np.nan, "shape_circularity": np.nan,
                    "shape_extent": np.nan, "shape_solidity": np.nan, 
                    "shape_centroid_x": np.nan, "shape_centroid_y": np.nan
                })
                continue
            
            # 3. Seleccionar la región más representativa
            candidates = [r for r in regions if 80 < r.area < img_uint8.size * 0.08]
            if candidates:
                lesion = max(candidates, key=lambda r: r.mean_intensity * r.solidity)
            else:
                lesion = max(regions, key=lambda r: r.area)
            
            # 4. Calcular métricas de forma
            perimeter = lesion.perimeter
            circ = (4 * np.pi * lesion.area / (perimeter**2)) if perimeter > 0 else np.nan
            
            features = {
                "shape_area": float(lesion.area),
                "shape_perimeter": float(perimeter),
                "shape_circularity": float(circ),
                "shape_extent": float(lesion.extent),
                "shape_solidity": float(lesion.solidity),
                "shape_centroid_x": float(lesion.centroid[1]), # Columna
                "shape_centroid_y": float(lesion.centroid[0])  # Fila
            }
            
            shape_features_list.append(features)
            
        except Exception as e:
            # Imprime el error exacto para saber qué falló en el procesamiento interno
            print(f"[!] Error en la fila {idx}: {e}")
            shape_features_list.append({
                "shape_area": np.nan, "shape_perimeter": np.nan, "shape_circularity": np.nan,
                "shape_extent": np.nan, "shape_solidity": np.nan, 
                "shape_centroid_x": np.nan, "shape_centroid_y": np.nan
            })
            
    # 5. Convertir a DataFrame y concatenar
    df_features = pd.DataFrame(shape_features_list)
    df_resultado = pd.concat([df.reset_index(drop=True), df_features], axis=1)
    
    return df_resultado

In [10]:
imgs_features_glcm_df = calcular_glcm_dataset(df)

In [ ]:
imgs_features_glcm_plus_shape_base = calcular_caracteristicas_forma_dataset(df)

/tmp/ipykernel_183474/2498333901.py:30: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  binary = remove_small_objects(binary, min_size=80)
/tmp/ipykernel_183474/2498333901.py:31: FutureWarning: Parameter `area_threshold` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_holes`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  binary = remove_small_holes(binary, area_threshold=200)
/tmp/ipykernel_183474/249833390

In [24]:
imgs_features_glcm_plus_shape_base

,Nombre_Archivo,path,clase,Ancho,Alto,Modo,shape_area,shape_perimeter,shape_circularity,shape_extent,shape_solidity,shape_centroid_x,shape_centroid_y
0,Normal-36.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,1025.0,402.988853,0.079314,0.187112,0.362191,4.434146,168.102439
1,Normal-332.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,96.0,54.041631,0.413071,0.410256,0.662069,468.083333,49.385417
2,Normal-840.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,145289.0,5833.010640,0.053661,0.554234,0.587824,242.787114,287.845783
3,Normal-1965.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,81.0,76.621320,0.173379,0.397059,0.632812,19.530864,0.444444
4,Normal-1304.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,131047.0,5290.508114,0.058836,0.499905,0.550856,263.485078,290.371020
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4195,Tuberculosis-366.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,4625.0,315.656854,0.583299,0.997627,0.998704,18.507459,450.641946
4196,Tuberculosis-555.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,186865.0,3784.237770,0.163976,0.712833,0.781221,306.204099,276.933679
4197,Tuberculosis-667.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,L,128.0,66.769553,0.360797,0.338624,0.711111,316.312500,222.656250
4198,Tuberculosis-217.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,20201.0,1007.641702,0.250018,0.368793,0.597928,62.625662,97.385278


In [23]:
imgs_features_glcm_df

,Nombre_Archivo,path,clase,Ancho,Alto,Modo,glcm_contrast,glcm_dissimilarity,glcm_homogeneity,glcm_energy,glcm_correlation
0,Normal-36.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,45.755570,3.082551,0.383555,0.067999,0.994814
1,Normal-332.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,10.787725,1.691166,0.505183,0.044947,0.995822
2,Normal-840.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,29.687370,3.071503,0.350901,0.046374,0.996419
3,Normal-1965.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,32.477240,3.048858,0.356249,0.042570,0.996241
4,Normal-1304.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,28.668172,3.198054,0.345249,0.034219,0.997167
...,...,...,...,...,...,...,...,...,...,...,...
4195,Tuberculosis-366.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,44.758398,2.735331,0.546848,0.072755,0.994984
4196,Tuberculosis-555.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,21.559046,1.459540,0.745914,0.511174,0.998140
4197,Tuberculosis-667.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,L,54.238006,2.721444,0.528639,0.167146,0.992908
4198,Tuberculosis-217.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,16.668138,2.044816,0.530283,0.056111,0.998379


In [27]:
imgs_features_glcm_df.to_csv("dataset_caracteristicas_glcm.csv", index=False)

In [ ]:
imgs_features_glcm_plus_shape_base.to_csv("", index=False)